# BIST Mum Formasyonu Tarayıcısı

**TradingView Scanner API** ile BIST hisselerinde **boğa** ve **ayı** mum formasyonu (candlestick pattern) taraması.

> Not: Tek başına alım/satım sinyali değildir — trend yönü, hacim ve direnç/destek teyidi gerekir.

## Endpoint

```
POST https://scanner.tradingview.com/turkey/scan?label-product=screener-stock
```

**Zorunlu header'lar:** `Origin` ve `Referer` (yoksa 403 dönebilir).

## İstek (Request) Alanları

| Alan | Açıklama |
|------|----------|
| `columns` | Dönecek alan listesi (pattern flag'leri dahil) |
| `filter` | Tek pattern için kısa yol · `[{"left":"Candle.Hammer","operation":"equal","right":1}]` |
| `filter2` | Nested AND/OR · birden fazla pattern'i OR ile bağlamak için |
| `sort` | Sıralama · `{sortBy, sortOrder}` |
| `range` | Sonuç dilimi · `[start, end]` |
| `markets` | Piyasa filtresi · `['turkey']` |
| `options.lang` | Yanıt dili · `'en'` veya `'tr'` |

## Operatörler

- `equal` → eşit (=) · pattern flag'i için `right: 1`
- `has` → listede içerir
- `has_none_of` → listede hiçbirini içermez
- `and` / `or` → `filter2` içinde nested mantıksal bağlaç

## Boğa Pattern'leri (Bullish Reversal)

| Pattern | Anlamı |
|---------|--------|
| `Candle.Hammer` | Çekiç · dip dönüş ihtimali |
| `Candle.InvertedHammer` | Ters çekiç · dip dönüş |
| `Candle.Engulfing.Bullish` | Yutan boğa · iki mumluk dönüş |
| `Candle.MorningStar` | Sabah yıldızı · üç mumluk dönüş |
| `Candle.Harami.Bullish` | Hamile · sıkışmadan kopuş ön habercisi |
| `Candle.3WhiteSoldiers` | Üç beyaz asker · güçlü trend dönüşü |
| `Candle.Marubozu.White` | Tam gövde beyaz · agresif alım |
| `Candle.Doji.Dragonfly` | Ejder doji · alıcı geri dönüşü |

## Ayı Pattern'leri (Bearish Reversal)

| Pattern | Anlamı |
|---------|--------|
| `Candle.ShootingStar` | Kayan yıldız · tepe dönüşü |
| `Candle.HangingMan` | Asılı adam · tepede uyarı |
| `Candle.Engulfing.Bearish` | Yutan ayı · iki mumluk dönüş |
| `Candle.EveningStar` | Akşam yıldızı · üç mumluk dönüş |
| `Candle.3BlackCrows` | Üç siyah karga · güçlü trend dönüşü |
| `Candle.Marubozu.Black` | Tam gövde siyah · agresif satış |

## Zaman Dilimi Son Ekleri

| Son ek | Anlam |
|--------|-------|
| (yok) | Günlük (1D) · default |
| `\|60` | 60 dakikalık · ör. `Candle.Hammer\|60` |
| `\|240` | 4 saatlik |
| `\|1W` | Haftalık |
| `\|1M` | Aylık |

Default zaman dilimi günlüktür. İntraday için `|60` / `|240`, üst zaman için `|1W` / `|1M` son eki.

## Yanıt (Response) Yapısı

`{ totalCount, data[] }` → `data`, bugün seçili pattern'lerden en az birini çeken BIST hisseleri.

Her `data[i]` öğesi:
- `s` → Tam sembol (örn. `'BIST:THYAO'`)
- `d[]` → `columns` sırasına göre değerler dizisi
  - `d[0]` name (sembol kodu)
  - `d[1]` close (son fiyat, TL)
  - `d[2]` change (yüzde değişim)
  - `d[3]` volume (günlük işlem hacmi)
  - `d[4]` market_cap_basic (piyasa değeri, TL)
  - `d[5+]` Pattern flag · `1` = bugün oluştu, `0` = oluşmadı

> Pattern alanları **integer 1/0** döner — boolean `true/false` DEĞİL. Filtre yazarken `right: 1` kullan.

## 1. Ortak Ayarlar

In [1]:
import requests

URL = "https://scanner.tradingview.com/turkey/scan?label-product=screener-stock"

HEADERS = {
    "content-type": "application/json",
    "origin": "https://www.tradingview.com",
    "referer": "https://www.tradingview.com/",
}

BASE_COLUMNS = ["name", "close", "change", "volume", "market_cap_basic"]

BULLISH_PATTERNS = [
    "Candle.Hammer",
    "Candle.InvertedHammer",
    "Candle.Engulfing.Bullish",
    "Candle.MorningStar",
    "Candle.Harami.Bullish",
    "Candle.3WhiteSoldiers",
    "Candle.Marubozu.White",
    "Candle.Doji.Dragonfly",
]

BEARISH_PATTERNS = [
    "Candle.ShootingStar",
    "Candle.HangingMan",
    "Candle.Engulfing.Bearish",
    "Candle.EveningStar",
    "Candle.3BlackCrows",
    "Candle.Marubozu.Black",
]

STOCK_ONLY = {
    "operator": "and",
    "operands": [
        {"operation": {"operator": "and", "operands": [
            {"expression": {"left": "type", "operation": "equal", "right": "stock"}},
            {"expression": {"left": "typespecs", "operation": "has", "right": ["common"]}},
        ]}},
        {"expression": {"left": "typespecs", "operation": "has_none_of", "right": ["pre-ipo"]}},
    ],
}

## 2. Tarama Fonksiyonu

Birden fazla pattern'i `filter2` içinde **OR** ile bağlıyoruz: hisse, listedeki pattern'lerden **en az birini** çekiyorsa sonuca giriyor.

`stock-only` zarfı (`type=stock`, `typespecs has 'common'`, `pre-ipo` hariç) ile ayrı bir **AND** dalında birleştiriyoruz — kısaca: stock olacak **VE** pattern'lerden biri tetiklenmiş olacak.

`tf` parametresi zaman dilimi son ekidir: `""` günlük, `"|60"` 60dk, `"|1W"` haftalık, vb.

In [2]:
def tara(patterns, tf="", limit=10):
    pattern_cols = [p + tf for p in patterns]
    columns = BASE_COLUMNS + pattern_cols

    pattern_or = {
        "operation": {
            "operator": "or",
            "operands": [
                {"expression": {"left": col, "operation": "equal", "right": 1}}
                for col in pattern_cols
            ],
        }
    }

    filter2 = {
        "operator": "and",
        "operands": STOCK_ONLY["operands"] + [pattern_or],
    }

    body = {
        "columns": columns,
        "sort": {"sortBy": "volume", "sortOrder": "desc"},
        "range": [0, limit],
        "markets": ["turkey"],
        "options": {"lang": "en"},
        "filter2": filter2,
    }
    r = requests.post(URL, headers=HEADERS, json=body)
    r.raise_for_status()
    return r.json(), pattern_cols


def yazdir(sonuc, pattern_cols, baslik):
    js, _ = sonuc, pattern_cols
    print(f"{baslik} · Toplam: {js['totalCount']}\n")
    print(f"{'Sembol':<15} {'Fiyat':>10} {'Değişim':>10} {'Hacim':>18}  Pattern(ler)")
    print("-" * 90)
    for row in js["data"]:
        sembol = row["s"]
        d = row["d"]
        _, close, change, volume, _ = d[:5]
        flags = d[5:]
        hits = [pc.split(".", 1)[1].split("|")[0] for pc, f in zip(pattern_cols, flags) if f == 1]
        print(f"{sembol:<15} {close:>10.2f} {change:>9.2f}% {volume:>18,.0f}  {', '.join(hits)}")

## 3. Boğa Pattern Taraması · Bugün dönüş sinyali çeken hisseler

In [ ]:
boga, cols = tara(BULLISH_PATTERNS, tf="", limit=10)
yazdir(boga, cols, "Boğa Pattern (günlük)")

## 4. Ayı Pattern Taraması · Bugün tepe dönüşü uyarısı çeken hisseler

In [ ]:
ayi, cols = tara(BEARISH_PATTERNS, tf="", limit=10)
yazdir(ayi, cols, "Ayı Pattern (günlük)")

## 5. Tek Pattern · Kısa Yol (`filter`)

Tek pattern arıyorsan `filter2` zarfına gerek yok — düz `filter` ile yazılabilir.

In [3]:
body = {
    "columns": BASE_COLUMNS + ["Candle.Hammer"],
    "filter": [{"left": "Candle.Hammer", "operation": "equal", "right": 1}],
    "sort": {"sortBy": "volume", "sortOrder": "desc"},
    "range": [0, 5],
    "markets": ["turkey"],
    "options": {"lang": "en"},
    "filter2": STOCK_ONLY,
}
r = requests.post(URL, headers=HEADERS, json=body)
r.raise_for_status()
js = r.json()

print(f"Hammer · Toplam: {js['totalCount']}\n")
print(f"{'Sembol':<15} {'Fiyat':>10} {'Değişim':>10} {'Hacim':>18}")
print("-" * 56)
for row in js["data"]:
    sembol = row["s"]
    _, close, change, volume, _, _ = row["d"]
    print(f"{sembol:<15} {close:>10.2f} {change:>9.2f}% {volume:>18,.0f}")

Hammer · Toplam: 0

Sembol               Fiyat    Değişim              Hacim
--------------------------------------------------------


## 6. Üst Zaman Diliminde Tarama · Haftalık Boğa Pattern

Pattern adlarına `|1W` son eki ekleyerek aynı taramayı haftalık mumlar üzerinden yapabiliriz.

In [4]:
haftalik, cols = tara(BULLISH_PATTERNS, tf="|1W", limit=10)
yazdir(haftalik, cols, "Boğa Pattern (haftalık)")

Boğa Pattern (haftalık) · Toplam: 26

Sembol               Fiyat    Değişim              Hacim  Pattern(ler)
------------------------------------------------------------------------------------------
BIST:ADESE            1.15      9.52%      1,040,932,209  Harami.Bullish
BIST:MEYSU           19.69     10.00%         42,995,874  Marubozu.White
BIST:ASTOR          243.60      9.98%         42,724,268  Marubozu.White
BIST:ETILR            5.33      9.90%         31,895,347  Marubozu.White
BIST:IMASM            3.93      1.29%         21,775,651  Harami.Bullish
BIST:NATEN            7.28      2.54%         20,978,496  Harami.Bullish
BIST:REEDR            8.02      1.78%         19,088,194  Harami.Bullish
BIST:DAGI             7.05      9.98%         15,972,274  Marubozu.White
BIST:KCAER           11.85      3.86%         14,857,129  Harami.Bullish
BIST:GESAN           49.88      7.78%         13,529,887  Marubozu.White


## Notlar

- Pattern alanları **integer 1/0** döner — boolean `true/false` DEĞİL. Filtre yazarken `right: 1` kullan.
- `filter` sadece **AND** bağlar; birden fazla pattern'i **OR** ile bağlamak için `filter2` içinde nested `or` operands aç.
- Default zaman dilimi günlüktür. İntraday için `|60` / `|240`, üst zaman için `|1W` / `|1M` son eki.
- Tek başına alım/satım sinyali değildir — trend yönü, hacim ve direnç/destek teyidi gerekir.
- `Origin` & `Referer` header'ları zorunlu — yoksa 403 dönebilir.